# Locomotion Mode CNN — WAWA Exoskeleton

## Pipeline Overview
1. **Decode** WAWA `.bin` files from STM32 SD card
2. **Train** CNN on CAMARGO public dataset (thigh IMU, 3 classes)
3. **Retrain** on WAWA data when collected

---

### WAWA Binary Format (from Navid's STM32 code)
```c
// Header: 0xDEADBEEF (4 bytes, once)
// Footer: 0xBEEFDEAD (4 bytes, once)
typedef struct __attribute__((packed)) {
    uint32_t timestamp;      // microseconds
    uint16_t adc[4];         // 4 EMG channels (raw ADC)
    int16_t  imu[10];        // acc(3) + gyro(3) + mag(3) + temp(1)
    int16_t  leftMotor[3];   // angle, velocity, torque?
    int16_t  rightMotor[3];  // angle, velocity, torque?
    int8_t   mode;           // knob switch (activity label)
} SensorData;                // 45 bytes per record
```

### WAWA Channels (19 useful for CNN)
| Channel | Source | Count |
|---------|--------|-------|
| IMU accel XYZ | tailbone | 3 |
| IMU gyro XYZ | tailbone | 3 |
| IMU mag XYZ | tailbone | 3 |
| EMG | 4 muscles | 4 |
| Left motor | angle/vel/torque | 3 |
| Right motor | angle/vel/torque | 3 |
| **Total** | | **19** |

## 0. Imports & Setup

In [1]:
import os, glob, struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

%matplotlib inline
plt.rcParams['figure.dpi'] = 140
plt.rcParams['font.size'] = 9

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
  GPU: NVIDIA RTX A5000


## 1. WAWA Binary Decoder

Decodes `.bin` files from the STM32 SD card into pandas DataFrames.

In [2]:
WAWA_MAGIC_START = 0xDEADBEEF
WAWA_MAGIC_END = 0xBEEFDEAD
WAWA_RECORD_FMT = '<I 4H 10h 3h 3h b'
WAWA_RECORD_SIZE = struct.calcsize(WAWA_RECORD_FMT)  # 45 bytes

WAWA_COLUMNS = [
    'time_us',
    'emg1', 'emg2', 'emg3', 'emg4',
    'acc_x', 'acc_y', 'acc_z',
    'gyr_x', 'gyr_y', 'gyr_z',
    'mag_x', 'mag_y', 'mag_z',
    'temp',
    'lmot_0', 'lmot_1', 'lmot_2',
    'rmot_0', 'rmot_1', 'rmot_2',
    'mode',
]


def decode_wawa_bin(filepath):
    """
    Decode a WAWA .bin file from the STM32 SD card.
    
    Returns:
        pd.DataFrame with columns from WAWA_COLUMNS
    """
    with open(filepath, 'rb') as f:
        data = f.read()
    
    # Check magic header
    magic = struct.unpack('<I', data[:4])[0]
    if magic != WAWA_MAGIC_START:
        raise ValueError(f'Bad magic: 0x{magic:08X} (expected 0xDEADBEEF)')
    
    # Check for end marker
    has_footer = False
    if len(data) >= 8:
        end_magic = struct.unpack('<I', data[-4:])[0]
        if end_magic == WAWA_MAGIC_END:
            has_footer = True
            data = data[:-4]  # strip footer
    
    # Parse records
    header_size = 4
    n_records = (len(data) - header_size) // WAWA_RECORD_SIZE
    
    rows = []
    for r in range(n_records):
        offset = header_size + r * WAWA_RECORD_SIZE
        rec = struct.unpack(WAWA_RECORD_FMT, data[offset:offset + WAWA_RECORD_SIZE])
        rows.append(rec)
    
    df = pd.DataFrame(rows, columns=WAWA_COLUMNS)
    
    # Add time in seconds
    df['time_s'] = df['time_us'] / 1e6
    
    duration = df['time_s'].iloc[-1] - df['time_s'].iloc[0]
    fs = 1.0 / df['time_s'].diff().median()
    
    print(f'Decoded: {filepath}')
    print(f'  Records: {n_records}, Duration: {duration:.2f}s, '
          f'Sample rate: {fs:.0f} Hz, Footer: {has_footer}')
    print(f'  Mode values: {sorted(df["mode"].unique())}')
    
    return df


def decode_wawa_to_csv(bin_path, csv_path=None):
    """Decode .bin to .csv for easy inspection."""
    df = decode_wawa_bin(bin_path)
    if csv_path is None:
        csv_path = bin_path.replace('.bin', '.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved: {csv_path}')
    return df


print(f'Record size: {WAWA_RECORD_SIZE} bytes')
print(f'Columns ({len(WAWA_COLUMNS)}): {WAWA_COLUMNS}')

Record size: 45 bytes
Columns (22): ['time_us', 'emg1', 'emg2', 'emg3', 'emg4', 'acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z', 'mag_x', 'mag_y', 'mag_z', 'temp', 'lmot_0', 'lmot_1', 'lmot_2', 'rmot_0', 'rmot_1', 'rmot_2', 'mode']


In [3]:
# === TEST: Decode a sample .bin file ===
# Update this path to point to your .bin file
BIN_FILE = os.path.expanduser(
    '~/repos/projects/exo-assist-pipeline/data/wawa/data1.bin')

# If the file exists, decode it
if os.path.exists(BIN_FILE):
    wawa_df = decode_wawa_bin(BIN_FILE)
    display(wawa_df.head())
    print(f'\nChannel stats:')
    display(wawa_df.describe().round(1))
else:
    print(f'File not found: {BIN_FILE}')
    print('Copy your .bin file to the path above, or update BIN_FILE.')

Decoded: /home/wxp/repos/projects/exo-assist-pipeline/data/wawa/data1.bin
  Records: 8827, Duration: 17.65s, Sample rate: 500 Hz, Footer: True
  Mode values: [np.int64(0)]


,time_us,emg1,emg2,emg3,emg4,acc_x,acc_y,acc_z,gyr_x,gyr_y,...,mag_z,temp,lmot_0,lmot_1,lmot_2,rmot_0,rmot_1,rmot_2,mode,time_s
0,2251302,611,827,965,549,-3576,11320,-11224,-8,98,...,-255,3072,63,0,0,0,0,0,0,2.251302
1,2253301,615,823,957,553,-3744,11224,-11208,182,-314,...,-255,3072,63,0,0,0,0,0,0,2.253301
2,2255301,605,820,959,553,-3704,10976,-11200,-159,69,...,-255,2960,63,0,0,0,0,0,0,2.255301
3,2257301,603,815,952,549,-3600,11320,-11280,-128,8,...,-257,3008,63,0,0,0,0,0,0,2.257301
4,2259301,603,809,946,541,-3496,11104,-11224,-161,-140,...,-257,2992,63,0,0,0,0,0,0,2.259301



Channel stats:


,time_us,emg1,emg2,emg3,emg4,acc_x,acc_y,acc_z,gyr_x,gyr_y,...,mag_z,temp,lmot_0,lmot_1,lmot_2,rmot_0,rmot_1,rmot_2,mode,time_s
count,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,...,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0,8827.0
mean,11077301.0,181.9,282.2,309.6,173.0,-3587.1,11197.7,-11175.6,-72.7,-31.8,...,-254.5,3031.8,63.0,0.0,0.0,0.0,0.0,0.0,0.0,11.1
std,5096559.5,39.6,47.4,58.1,34.7,114.8,115.7,98.2,225.1,298.2,...,13.4,54.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.1
min,2251302.0,17.0,106.0,118.0,24.0,-4040.0,9976.0,-11672.0,-916.0,-1203.0,...,-511.0,2880.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,2.3
25%,6664301.0,165.0,264.0,285.0,157.0,-3664.0,11120.0,-11240.0,-231.0,-233.0,...,-258.0,2992.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,6.7
50%,11077301.0,173.0,275.0,297.0,167.0,-3584.0,11200.0,-11176.0,-72.0,-32.0,...,-255.0,3024.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,11.1
75%,15490301.0,189.0,287.0,315.0,181.0,-3512.0,11272.0,-11112.0,89.0,167.0,...,-251.0,3072.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,15.5
max,19903301.0,619.0,827.0,965.0,553.0,-3144.0,11856.0,-10720.0,716.0,997.0,...,-1.0,3184.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,19.9


In [ ]:
# === Visualize WAWA data ===
if os.path.exists(BIN_FILE):
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    t = wawa_df['time_s'] - wawa_df['time_s'].iloc[0]
    
    # EMG
    for ch in ['emg1', 'emg2', 'emg3', 'emg4']:
        axes[0].plot(t, wawa_df[ch], lw=0.4, alpha=0.7, label=ch)
    axes[0].set_title('EMG (raw ADC)', fontweight='bold')
    axes[0].legend(ncol=4, fontsize=7); axes[0].grid(alpha=0.3)
    
    # IMU Accel + Gyro
    for ch in ['acc_x', 'acc_y', 'acc_z']:
        axes[1].plot(t, wawa_df[ch], lw=0.5, alpha=0.7, label=ch)
    axes[1].set_title('Accelerometer', fontweight='bold')
    axes[1].legend(ncol=3, fontsize=7); axes[1].grid(alpha=0.3)
    
    for ch in ['gyr_x', 'gyr_y', 'gyr_z']:
        axes[2].plot(t, wawa_df[ch], lw=0.5, alpha=0.7, label=ch)
    axes[2].set_title('Gyroscope', fontweight='bold')
    axes[2].legend(ncol=3, fontsize=7); axes[2].grid(alpha=0.3)
    
    # Motor data
    for ch in ['lmot_0', 'lmot_1', 'lmot_2', 'rmot_0', 'rmot_1', 'rmot_2']:
        axes[3].plot(t, wawa_df[ch], lw=0.5, alpha=0.7, label=ch)
    axes[3].set_title('Motor Encoders', fontweight='bold')
    axes[3].legend(ncol=6, fontsize=7); axes[3].grid(alpha=0.3)
    axes[3].set_xlabel('Time (s)')
    
    fig.suptitle('WAWA Raw Sensor Data', fontsize=13, fontweight='bold')
    fig.tight_layout()
    plt.show()

---

## 2. Train CNN on CAMARGO Data

Using thigh IMU (6 channels) from the CAMARGO public dataset as a proxy
for the WAWA tailbone IMU. 3 classes: walk / ramp / stair.

### Configuration

In [ ]:
CAMARGO_DIR = os.path.expanduser(
    '~/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18')

THIGH_COLS = ['thigh_Accel_X', 'thigh_Accel_Y', 'thigh_Accel_Z',
              'thigh_Gyro_X', 'thigh_Gyro_Y', 'thigh_Gyro_Z']
N_CH = len(THIGH_COLS)

FS = 200
WIN_SEC = 0.2
STEP_SEC = 0.1
WIN = int(FS * WIN_SEC)
STEP = int(FS * STEP_SEC)

LABEL_MAP = {'levelground': 0, 'ramp': 1, 'stair': 2}
LABELS = ['walk', 'ramp', 'stair']
N_CLASSES = len(LABELS)

print(f'Channels: {N_CH}')
print(f'Window: {WIN} samples = {WIN_SEC*1000:.0f}ms')
print(f'Step: {STEP} samples = {STEP_SEC*1000:.0f}ms')
print(f'Classes: {LABELS}')

### 3. Load CAMARGO Trials

In [ ]:
trials = []
for mode in ['levelground', 'ramp', 'stair']:
    csvs = sorted(glob.glob(os.path.join(CAMARGO_DIR, mode, 'imu', '*.csv')))
    for f in csvs:
        df = pd.read_csv(f)
        data = df[THIGH_COLS].values.astype(np.float32)
        trials.append((data, mode, os.path.basename(f)))

print(f'Loaded {len(trials)} trials: {Counter([t[1] for t in trials])}')
print(f'Sample shape: {trials[0][0].shape}')

# Duration stats
for mode in LABEL_MAP:
    durs = [t[0].shape[0]/FS for t in trials if t[1] == mode]
    print(f'  {mode}: {np.mean(durs):.1f} ± {np.std(durs):.1f}s per trial')

### 4. Visualize Raw IMU

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8))
ch_names = ['Acc X', 'Acc Y', 'Acc Z', 'Gyr X', 'Gyr Y', 'Gyr Z']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00', '#984ea3', '#a65628']

for ax, mode in zip(axes, ['levelground', 'ramp', 'stair']):
    trial = next(t for t in trials if t[1] == mode)
    data = trial[0]
    t = np.arange(data.shape[0]) / FS
    for ch in range(N_CH):
        ax.plot(t, data[:, ch], lw=0.5, alpha=0.7, color=colors[ch], label=ch_names[ch])
    ax.set_title(f'{mode}', fontsize=10, fontweight='bold')
    ax.set_ylabel('IMU'); ax.grid(alpha=0.3)

axes[0].legend(ncol=6, fontsize=7, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.tight_layout(); plt.show()

### 5. Create Sliding Windows

In [ ]:
Xw, yw = [], []
for data, mode, _ in trials:
    lab = LABEL_MAP[mode]
    for s in range(0, data.shape[0] - WIN + 1, STEP):
        Xw.append(data[s:s+WIN].T)  # (6, 40)
        yw.append(lab)

X = np.array(Xw, dtype=np.float32)
y = np.array(yw, dtype=np.int64)

print(f'X: {X.shape}  (windows, channels, time)')
for i, name in enumerate(LABELS):
    c = (y == i).sum()
    print(f'  {name}: {c} ({c/len(y)*100:.1f}%)')

### 6. Normalize & Split

In [ ]:
X_mean = X.mean(axis=(0, 2), keepdims=True)
X_std = X.std(axis=(0, 2), keepdims=True) + 1e-8
X_norm = (X - X_mean) / X_std

Xtr, Xte, ytr, yte = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {Xtr.shape[0]}, Test: {Xte.shape[0]}')

### 7. CNN Model

```
Input (6, 40)
  → Conv1d(6→32, k=5) + BN + ReLU + MaxPool    → (32, 20)
  → Conv1d(32→64, k=5) + BN + ReLU + MaxPool   → (64, 10)
  → Conv1d(64→128, k=3) + BN + ReLU + AvgPool   → (128,)
  → Dropout + FC(128→64) + ReLU + FC(64→3)
```

In [ ]:
class LocomotionCNN(nn.Module):
    def __init__(self, n_ch=6, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_ch, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1))

model = LocomotionCNN(N_CH, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

### 8. Train

In [ ]:
train_dl = DataLoader(
    TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), 64, shuffle=True)
test_dl = DataLoader(
    TensorDataset(torch.from_numpy(Xte), torch.from_numpy(yte)), 128)

counts = np.bincount(ytr)
weights = torch.FloatTensor(len(counts) / (len(counts) * counts)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=weights)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

N_EPOCHS = 40
hist = {'tl': [], 'vl': [], 'va': []}

for ep in range(N_EPOCHS):
    model.train(); tl = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        tl += loss.item() * len(xb)
    tl /= len(train_dl.dataset)

    model.eval(); vl = cor = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            vl += criterion(out, yb).item() * len(xb)
            cor += (out.argmax(1) == yb).sum().item()
    vl /= len(test_dl.dataset)
    va = cor / len(test_dl.dataset)
    scheduler.step(vl)

    hist['tl'].append(tl); hist['vl'].append(vl); hist['va'].append(va)
    if (ep+1) % 5 == 0 or ep == 0:
        print(f'Ep {ep+1:3d}/{N_EPOCHS}  train={tl:.4f}  test={vl:.4f}  acc={va:.3f}')

print(f'\n>>> Best accuracy: {max(hist["va"]):.3f}')

### 9. Training Curves

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(hist['tl'], label='Train'); a1.plot(hist['vl'], label='Test')
a1.set_xlabel('Epoch'); a1.set_ylabel('Loss'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(hist['va'], color='green')
a2.axhline(max(hist['va']), color='green', ls='--', alpha=0.3)
a2.set_xlabel('Epoch'); a2.set_ylabel('Accuracy')
a2.set_title(f'Best: {max(hist["va"]):.3f}'); a2.grid(alpha=0.3)
fig.tight_layout(); plt.show()

### 10. Confusion Matrix

In [ ]:
model.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        labels.extend(yb.numpy())

print(classification_report(labels, preds, target_names=LABELS))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap='Blues')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=13,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(LABELS); ax.set_yticklabels(LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix', fontweight='bold')
fig.tight_layout(); plt.show()

### 11. Save Model

In [ ]:
save_dir = os.path.expanduser('~/repos/projects/exo-assist-pipeline/models')
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'config': {'n_ch': N_CH, 'n_classes': N_CLASSES, 'win': WIN, 'fs': FS},
    'labels': LABELS,
    'norm': {'mean': X_mean, 'std': X_std},
    'accuracy': max(hist['va']),
    'history': hist,
}, os.path.join(save_dir, 'locomotion_cnn_v1.pt'))

print(f'Saved to {save_dir}/locomotion_cnn_v1.pt')
print(f'Accuracy: {max(hist["va"]):.3f}')

---

## 12. Batch Decode WAWA Files

When you have multiple `.bin` files from data collection sessions,
run this cell to decode them all to CSV.

In [ ]:
WAWA_DATA_DIR = os.path.expanduser(
    '~/repos/projects/exo-assist-pipeline/data/wawa')

bin_files = sorted(glob.glob(os.path.join(WAWA_DATA_DIR, '*.bin')))
print(f'Found {len(bin_files)} .bin files')

for bf in bin_files:
    try:
        decode_wawa_to_csv(bf)
    except Exception as e:
        print(f'  ERROR: {bf}: {e}')

---

## Next Steps

1. **Collect WAWA data** — walk/stairs/ramp with knob labels, EMG + IMU + motors
2. **Decode** — run cell 12 to batch convert `.bin` → `.csv`
3. **Retrain CNN** — swap CAMARGO loader with WAWA CSV loader, expand to 19 channels
4. **5-class expansion** — separate ascent/descent using `mode` knob values
5. **Add more subjects** — download more CAMARGO subjects or collect from lab members
6. **Integrate with gait phase** — combine Yang's φ(t) with CNN output
7. **Deploy** — export to ONNX/TFLite for STM32 microcontroller inference
8. **Confirm with Navid:**
   - What are `leftMotor[0,1,2]` and `rightMotor[0,1,2]`? (angle, velocity, torque?)
   - What knob positions map to which activities?
   - Are IMU values raw ADC or scaled to physical units?